In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# Self querying

self-query retriever 要求您安装 lark 软件包。

In [2]:
# !pip install lark==1.1.9

## 索引

In [3]:
# from langchain_chroma import Chroma # 教程里是这样写的，但这种写法已经被弃用
from langchain_community.vectorstores import Chroma # 使用这个
# from langchain_community.vectorstores import Qdrant
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

docs = [
    Document(
        page_content="一群科学家带回了恐龙，混乱随之而来",
        metadata={"year": 1993, "rating": 7.7, "genre": "科幻"},
    ),
    Document(
        page_content="莱昂纳多·迪卡普里奥在一个梦中的梦中迷失...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="一位心理学家/侦探在一系列梦中迷失，梦中有梦，《盗梦空间》借用了这个创意",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="一群正常身材的女性极其健康，一些男人对她们心生爱慕",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="玩具们活了过来，并乐在其中",
        metadata={"year": 1995, "genre": "动画"},
    ),
    Document(
        page_content="三个人进入了区域，三个人走出了区域",
        metadata={
            "year": 1979,
            "rating": 9.9,
            "director": "Andrei Tarkovsky",
            "genre": "科幻",
        },
    ),
]

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(docs, embeddings)
# vectorstore = Qdrant.from_documents(
#     docs,
#     embeddings,
#     location=":memory:",  # 本地模式，只使用内存存储
#     collection_name="my_documents",  # 集合名称为“my_documents”
# )

In [4]:
# 相似性搜索
query = "关于恐龙的电影"
vectorstore.similarity_search(query)

[Document(page_content='一群科学家带回了恐龙，混乱随之而来', metadata={'genre': '科幻', 'rating': 7.7, 'year': 1993}),
 Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995}),
 Document(page_content='一位心理学家/侦探在一系列梦中迷失，梦中有梦，《盗梦空间》借用了这个创意', metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006}),
 Document(page_content='莱昂纳多·迪卡普里奥在一个梦中的梦中迷失...', metadata={'director': 'Christopher Nolan', 'rating': 8.2, 'year': 2010})]

In [5]:
# 带分数的相似性搜索
query = "关于恐龙的电影"
vectorstore.similarity_search_with_score(query)

[(Document(page_content='一群科学家带回了恐龙，混乱随之而来', metadata={'genre': '科幻', 'rating': 7.7, 'year': 1993}),
  0.32052403688430786),
 (Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995}),
  0.4249217212200165),
 (Document(page_content='一位心理学家/侦探在一系列梦中迷失，梦中有梦，《盗梦空间》借用了这个创意', metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006}),
  0.45001912117004395),
 (Document(page_content='莱昂纳多·迪卡普里奥在一个梦中的梦中迷失...', metadata={'director': 'Christopher Nolan', 'rating': 8.2, 'year': 2010}),
  0.4504508376121521)]

In [6]:
# 带分数的相似性搜索
query = ""
vectorstore.similarity_search_with_score(query)

[(Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995}),
  0.6423704624176025),
 (Document(page_content='莱昂纳多·迪卡普里奥在一个梦中的梦中迷失...', metadata={'director': 'Christopher Nolan', 'rating': 8.2, 'year': 2010}),
  0.6495233774185181),
 (Document(page_content='一群科学家带回了恐龙，混乱随之而来', metadata={'genre': '科幻', 'rating': 7.7, 'year': 1993}),
  0.6521576642990112),
 (Document(page_content='三个人进入了区域，三个人走出了区域', metadata={'director': 'Andrei Tarkovsky', 'genre': '科幻', 'rating': 9.9, 'year': 1979}),
  0.6635635495185852)]

## 创建检索器

In [7]:
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_openai import OpenAI, ChatOpenAI

metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="电影的类型",
        type="string 或 list[string]",
    ),
    AttributeInfo(
        name="year",
        description="电影上映的年份",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="电影导演的名字",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="电影的评分（1-10）",
        type="float"
    ),
]
document_content_description = "电影的简短总结"
llm = ChatOpenAI(temperature=0)
retriever = SelfQueryRetriever.from_llm(
    llm, 
    vectorstore, 
    document_content_description, 
    metadata_field_info, 
    verbose=True
)

## 测试检索器

In [8]:
# 本例只指定了一个过滤器
retriever.invoke("我想看一部评分高于 8.5 分的电影")

[Document(page_content='三个人进入了区域，三个人走出了区域', metadata={'director': 'Andrei Tarkovsky', 'genre': '科幻', 'rating': 9.9, 'year': 1979}),
 Document(page_content='一位心理学家/侦探在一系列梦中迷失，梦中有梦，《盗梦空间》借用了这个创意', metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006})]

In [9]:
# 相似性搜索
vectorstore.similarity_search_with_score("我想看一部评分高于 8.5 分的电影")

[(Document(page_content='一位心理学家/侦探在一系列梦中迷失，梦中有梦，《盗梦空间》借用了这个创意', metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006}),
  0.41993841528892517),
 (Document(page_content='莱昂纳多·迪卡普里奥在一个梦中的梦中迷失...', metadata={'director': 'Christopher Nolan', 'rating': 8.2, 'year': 2010}),
  0.4616195261478424),
 (Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995}),
  0.4869420528411865),
 (Document(page_content='一群正常身材的女性极其健康，一些男人对她们心生爱慕', metadata={'director': 'Greta Gerwig', 'rating': 8.3, 'year': 2019}),
  0.5120815634727478)]

In [10]:
# 本例指定了一个查询和一个过滤器
retriever.invoke("Greta Gerwig 曾执导过任何关于女性的电影吗？")

[Document(page_content='一群正常身材的女性极其健康，一些男人对她们心生爱慕', metadata={'director': 'Greta Gerwig', 'rating': 8.3, 'year': 2019})]

In [11]:
# 本例指定了一个复合过滤器
retriever.invoke("什么是高分（8.5 分以上）科幻电影？")

[Document(page_content='一位心理学家/侦探在一系列梦中迷失，梦中有梦，《盗梦空间》借用了这个创意', metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006}),
 Document(page_content='三个人进入了区域，三个人走出了区域', metadata={'director': 'Andrei Tarkovsky', 'genre': '科幻', 'rating': 9.9, 'year': 1979})]

In [12]:
# 这个示例指定了一个查询和复合过滤器
retriever.invoke(
    "1990年后且2005年前的电影，最好是关于玩具的动画片"
)

[]

In [13]:
retriever = SelfQueryRetriever.from_llm(
    ChatOpenAI(model_name="gpt-4-turbo", temperature=0),
    vectorstore,
    document_content_description,
    metadata_field_info,
    enable_limit=True,
)

In [14]:
# 这个示例指定了一个查询和复合过滤器
retriever.invoke(
    "1990年后且2005年前的电影，最好是关于玩具的动画片"
)

[Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995}),
 Document(page_content='一群科学家带回了恐龙，混乱随之而来', metadata={'genre': '科幻', 'rating': 7.7, 'year': 1993})]

过滤器 K

In [15]:
retriever = SelfQueryRetriever.from_llm(
    ChatOpenAI(model_name="gpt-4-turbo", temperature=0),
    vectorstore,
    document_content_description,
    metadata_field_info,
    enable_limit=True,
)

In [16]:
# 这个示例仅指定了一个相关的查询
retriever.invoke("关于恐龙的两部电影是什么")

[Document(page_content='一群科学家带回了恐龙，混乱随之而来', metadata={'genre': '科幻', 'rating': 7.7, 'year': 1993}),
 Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995})]

In [17]:
# 这个示例仅指定了一个相关的查询
retriever.invoke("关于恐龙的三部电影是什么")

[Document(page_content='一群科学家带回了恐龙，混乱随之而来', metadata={'genre': '科幻', 'rating': 7.7, 'year': 1993}),
 Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995}),
 Document(page_content='莱昂纳多·迪卡普里奥在一个梦中的梦中迷失...', metadata={'director': 'Christopher Nolan', 'rating': 8.2, 'year': 2010})]

## 使用LCEL从头开始构建

```json
{
    "query": "",
    "filter": "gt(\"rating\", 8.5)"
}
```

```
{
  "output": {
    "query": " ",
    "filter": {
      "comparator": "gt",
      "attribute": "rating",
      "value": 8.5
    }
  }
}
```

```json
{
    "query": "女性",
    "filter": "eq(\"director\", \"Greta Gerwig\")"
}
```

```
{
  "output": {
    "query": "女性",
    "filter": {
      "comparator": "eq",
      "attribute": "director",
      "value": "Greta Gerwig"
    }
  }
}
```

```json
{
    "query": "science fiction",
    "filter": "gte(\"rating\", 8.5)"
}
```

```
{
  "output": {
    "query": "science fiction",
    "filter": {
      "comparator": "gte",
      "attribute": "rating",
      "value": 8.5
    }
  }
}
```

```json
{
    "query": "玩具",
    "filter": "and(and(gte(\"year\", 1990), lt(\"year\", 2005)), eq(\"genre\", \"动画片\"))"
}
```

```
{
  "output": {
    "query": "玩具",
    "filter": {
      "operator": "and",
      "arguments": [
        {
          "operator": "and",
          "arguments": [
            {
              "comparator": "gte",
              "attribute": "year",
              "value": 1990
            },
            {
              "comparator": "lt",
              "attribute": "year",
              "value": 2005
            }
          ]
        },
        {
          "comparator": "eq",
          "attribute": "genre",
          "value": "动画片"
        }
      ]
    }
  }
}
```

### Query Constructor

In [18]:
from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser

In [19]:
query_constructor.invoke("关于恐龙的两部电影是什么")

StructuredQuery(query='恐龙', filter=None, limit=None)

In [20]:
query_constructor.invoke("我想看一部评分高于 8.5 分的电影")

StructuredQuery(query=' ', filter=Comparison(comparator=<Comparator.GT: 'gt'>, attribute='rating', value=8.5), limit=None)

In [21]:
type(prompt)

langchain_core.prompts.few_shot.FewShotPromptTemplate

In [22]:
type(llm)

langchain_openai.chat_models.base.ChatOpenAI

In [23]:
type(output_parser)

langchain.chains.query_constructor.base.StructuredQueryOutputParser

In [24]:
type(query_constructor.invoke("我想看一部评分高于 8.5 分的电影"))

langchain_core.structured_query.StructuredQuery

#### prompt

In [25]:
prompt

FewShotPromptTemplate(input_variables=['query'], examples=[{'i': 1, 'data_source': '```json\n{{\n    "content": "Lyrics of a song",\n    "attributes": {{\n        "artist": {{\n            "type": "string",\n            "description": "Name of the song artist"\n        }},\n        "length": {{\n            "type": "integer",\n            "description": "Length of the song in seconds"\n        }},\n        "genre": {{\n            "type": "string",\n            "description": "The song genre, one of "pop", "rock" or "rap""\n        }}\n    }}\n}}\n```', 'user_query': 'What are songs by Taylor Swift or Katy Perry about teenage romance under 3 minutes long in the dance pop genre', 'structured_request': '```json\n{{\n    "query": "teenager love",\n    "filter": "and(or(eq(\\"artist\\", \\"Taylor Swift\\"), eq(\\"artist\\", \\"Katy Perry\\")), lt(\\"length\\", 180), eq(\\"genre\\", \\"pop\\"))"\n}}\n```'}, {'i': 2, 'data_source': '```json\n{{\n    "content": "Lyrics of a song",\n    "attri

In [26]:
# 查看提示

In [27]:
print(prompt.format(query="假问题"))

Your goal is to structure the user's query to match the request schema provided below.

<< Structured Request Schema >>
When responding use a markdown code snippet with a JSON object formatted in the following schema:

```json
{
    "query": string \ text string to compare to document contents
    "filter": string \ logical condition statement for filtering documents
}
```

The query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.

A logical condition statement is composed of one or more comparison and logical operation statements.

A comparison statement takes the form: `comp(attr, val)`:
- `comp` (eq | ne | gt | gte | lt | lte | contain | like | in | nin): comparator
- `attr` (string):  name of attribute to apply the comparison to
- `val` (string): is the comparison value

A logical operation statement takes the form `op(statement1, statement2, ...)`:
- `op` (and | or | not

In [28]:
prompt.example_prompt

PromptTemplate(input_variables=['data_source', 'i', 'structured_request', 'user_query'], template='<< Example {i}. >>\nData Source:\n{data_source}\n\nUser Query:\n{user_query}\n\nStructured Request:\n{structured_request}\n')

In [29]:
print(prompt.example_prompt.template)

<< Example {i}. >>
Data Source:
{data_source}

User Query:
{user_query}

Structured Request:
{structured_request}



In [30]:
# 吕克·贝松导演的90年代关于出租车司机的科幻电影有哪些
query_constructor.invoke(
    {
        "query": "What are some sci-fi movies from the 90's directed by Luc Besson about taxi drivers"
    }
)

StructuredQuery(query='taxi drivers', filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='genre', value='sci-fi'), Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='year', value=1990), Comparison(comparator=<Comparator.LT: 'lt'>, attribute='year', value=2000)]), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='Luc Besson')]), limit=None)

```json
{
    "query": "taxi drivers",
    "filter": "and(eq(\"genre\", \"sci-fi\"), and(gte(\"year\", 1990), lt(\"year\", 2000)), eq(\"director\", \"Luc Besson\"))"
}
```

```
{
  "output": {
    "query": "taxi drivers",
    "filter": {
      "operator": "and",
      "arguments": [
        {
          "comparator": "eq",
          "attribute": "genre",
          "value": "sci-fi"
        },
        {
          "operator": "and",
          "arguments": [
            {
              "comparator": "gte",
              "attribute": "year",
              "value": 1990
            },
            {
              "comparator": "lt",
              "attribute": "year",
              "value": 2000
            }
          ]
        },
        {
          "comparator": "eq",
          "attribute": "director",
          "value": "Luc Besson"
        }
      ]
    }
  }
}
```

### Query Translator

In [31]:
# from langchain.retrievers.self_query.chroma import ChromaTranslator # 教程里是这样写的，但这种写法已经被弃用
from langchain_community.query_constructors.chroma import ChromaTranslator # 使用这个
# from langchain_community.query_constructors.qdrant import QdrantTranslator

retriever = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=vectorstore,
    structured_query_translator=ChromaTranslator(),
    # structured_query_translator=QdrantTranslator("metadata"),
)

In [32]:
retriever.structured_query_translator

In [33]:
# 这个示例指定了一个查询和复合过滤器
retriever.invoke(
    "1990年后且2005年前的电影，最好是关于玩具的动画片"
)

ValueError: Received disallowed comparator like. Allowed comparators are [<Comparator.EQ: 'eq'>, <Comparator.NE: 'ne'>, <Comparator.GT: 'gt'>, <Comparator.GTE: 'gte'>, <Comparator.LT: 'lt'>, <Comparator.LTE: 'lte'>]

报错：
```
ValueError: Received disallowed comparator like. Allowed comparators are [<Comparator.EQ: 'eq'>, <Comparator.NE: 'ne'>, <Comparator.GT: 'gt'>, <Comparator.GTE: 'gte'>, <Comparator.LT: 'lt'>, <Comparator.LTE: 'lte'>]
```

参考：
[https://github.com/langchain-ai/langchain/issues/3788](https://github.com/langchain-ai/langchain/issues/3788)

报错原因：ChromaDB 只允许 6种 操作符

```
allowed_comparators = [
        Comparator.EQ,
        Comparator.NE,
        Comparator.GT,
        Comparator.GTE,
        Comparator.LT,
        Comparator.LTE,
    ]
```

In [34]:
ChromaTranslator??

Init signature: ChromaTranslator()
Source:        
class ChromaTranslator(Visitor):
    """Translate `Chroma` internal query language elements to valid filters."""

    allowed_operators = [Operator.AND, Operator.OR]
    """Subset of allowed logical operators."""
    allowed_comparators = [
        Comparator.EQ,
        Comparator.NE,
        Comparator.GT,
        Comparator.GTE,
        Comparator.LT,
        Comparator.LTE,
    ]
    """Subset of allowed logical comparators."""

    def _format_func(self, func: Union[Operator, Comparator]) -> str:
        self._validate_func(func)
        return f"${func.value}"

    def visit_operation(self, operation: Operation) -> Dict:
        args = [arg.accept(self) for arg in operation.arguments]
        return {self._format_func(operation.operator): args}

    def visit_comparison(self, comparison: Comparison) -> Dict:
        return {
            comparison.attribute: {
                self._format_func(comparison.comparator): comparison.v

#### 修复错误

In [35]:
# Query Constructor

from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

# 定义允许的比较符列表
allowed_comparators = [
    "eq",  # 等于（数字、字符串、布尔值）
    "ne",  # 不等于（数字、字符串、布尔值）
    "gt",  # 大于（数字）
    "gte",  # 大于或等于（数字）
    "lt",  # 小于（数字）
    "lte",  # 小于或等于（数字）
]

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
    allowed_comparators=allowed_comparators,
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser

In [36]:
print(prompt.format(query="假问题"))

Your goal is to structure the user's query to match the request schema provided below.

<< Structured Request Schema >>
When responding use a markdown code snippet with a JSON object formatted in the following schema:

```json
{
    "query": string \ text string to compare to document contents
    "filter": string \ logical condition statement for filtering documents
}
```

The query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.

A logical condition statement is composed of one or more comparison and logical operation statements.

A comparison statement takes the form: `comp(attr, val)`:
- `comp` (eq | ne | gt | gte | lt | lte): comparator
- `attr` (string):  name of attribute to apply the comparison to
- `val` (string): is the comparison value

A logical operation statement takes the form `op(statement1, statement2, ...)`:
- `op` (and | or | not): logical operator
- `state

In [37]:
# Query Translator

from langchain_community.query_constructors.chroma import ChromaTranslator

retriever = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=vectorstore,
    structured_query_translator=ChromaTranslator(),
)

In [38]:
# 这个示例指定了一个查询和复合过滤器
retriever.invoke(
    "1990年后且2005年前的电影，最好是关于玩具的动画片"
)

[]

```json
{
    "query": "玩具",
    "filter": "and(and(gte(\"year\", 1990), lt(\"year\", 2005)), eq(\"genre\", \"动画片\"))"
}
```

#### 优化 —— 换模型

In [39]:
# Query Constructor

from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

# 定义允许的比较符列表
allowed_comparators = [
    "eq",  # 等于（数字、字符串、布尔值）
    "ne",  # 不等于（数字、字符串、布尔值）
    "gt",  # 大于（数字）
    "gte",  # 大于或等于（数字）
    "lt",  # 小于（数字）
    "lte",  # 小于或等于（数字）
]

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
    allowed_comparators=allowed_comparators,
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | ChatOpenAI(model_name="gpt-4-turbo", temperature=0) | output_parser


# Query Translator

from langchain_community.query_constructors.chroma import ChromaTranslator

retriever = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=vectorstore,
    structured_query_translator=ChromaTranslator(),
)

In [40]:
# 这个示例指定了一个查询和复合过滤器
retriever.invoke(
    "1990年后且2005年前的电影，最好是关于玩具的动画片"
)

[Document(page_content='玩具们活了过来，并乐在其中', metadata={'genre': '动画', 'year': 1995}),
 Document(page_content='一群科学家带回了恐龙，混乱随之而来', metadata={'genre': '科幻', 'rating': 7.7, 'year': 1993})]

```json
{
    "query": "玩具的动画片",
    "filter": "and(gt(\"year\", 1990), lt(\"year\", 2005))"
}
```

#### 分析底层

In [41]:
structured_query = query_constructor.invoke("1990年后且2005年前的电影，最好是关于玩具的动画片")
structured_query

StructuredQuery(query='玩具 动画片', filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.GT: 'gt'>, attribute='year', value=1990), Comparison(comparator=<Comparator.LT: 'lt'>, attribute='year', value=2005)]), limit=None)

In [42]:
structured_query.query

'玩具 动画片'

In [43]:
structured_query.filter

Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.GT: 'gt'>, attribute='year', value=1990), Comparison(comparator=<Comparator.LT: 'lt'>, attribute='year', value=2005)])

In [47]:
type(structured_query.filter)

langchain_core.structured_query.Operation

In [46]:
structured_query.limit

In [44]:
structured_query_translator = ChromaTranslator()
structured_query_translator

In [45]:
structured_query_translator.visit_structured_query(structured_query)

('玩具 动画片',
 {'filter': {'$and': [{'year': {'$gt': 1990}}, {'year': {'$lt': 2005}}]}})

In [49]:
structured_query.filter.accept(structured_query_translator)

{'$and': [{'year': {'$gt': 1990}}, {'year': {'$lt': 2005}}]}

In [50]:
structured_query.filter.accept??

Signature: structured_query.filter.accept(visitor: 'Visitor') -> 'Any'
Source:   
    def accept(self, visitor: Visitor) -> Any:
        """Accept a visitor.

        Args:
            visitor: visitor to accept

        Returns:
            result of visiting
        """
        return getattr(visitor, f"visit_{_to_snake_case(self.__class__.__name__)}")(
            self
        )
File:      d:\software\anaconda3\envs\py310_llmappcourse2\lib\site-packages\langchain_core\structured_query.py
Type:      method

In [51]:
structured_query.filter

Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.GT: 'gt'>, attribute='year', value=1990), Comparison(comparator=<Comparator.LT: 'lt'>, attribute='year', value=2005)])

In [52]:
type(structured_query.filter)

langchain_core.structured_query.Operation

In [53]:
structured_query_translator.visit_operation(structured_query.filter)

{'$and': [{'year': {'$gt': 1990}}, {'year': {'$lt': 2005}}]}

In [54]:
structured_query.filter.arguments

[Comparison(comparator=<Comparator.GT: 'gt'>, attribute='year', value=1990),
 Comparison(comparator=<Comparator.LT: 'lt'>, attribute='year', value=2005)]

In [55]:
structured_query.filter.arguments[0]

Comparison(comparator=<Comparator.GT: 'gt'>, attribute='year', value=1990)

In [56]:
structured_query_translator.visit_comparison(structured_query.filter.arguments[0])

{'year': {'$gt': 1990}}

In [57]:
type(structured_query_translator.visit_comparison(structured_query.filter.arguments[0]))

dict

In [58]:
structured_query_translator.visit_comparison(structured_query.filter.arguments[1])

{'year': {'$lt': 2005}}